In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import requests
import os
from datetime import date, timedelta
import datetime

## Part 1

The source of this data comes from NASA's NeoWs (Near Earth Object Web Service). When requesting data from the API, the end data can be a maximum of seven days after the start date. The raw data contains lots of information about asteriods that are considered Near Earth Objects (NEOs). In this project, only seven variable from the raw data are considered, these seven variables include: Approach Date, Miss Distance, Relative Velocity, Object Id (primary key), Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag.

Every approach date contians approach data from an object on that day. Because of this reason, the data is read in as two seperate DataFrames that are merged on approach date. Approach date is used as the key for this merge, however, moving forward, Object Id is used as the primary key as it is more practical.

The first DataFrame created contains the Approach Date, Miss Distance, and Relative Velocity for every NEO observed on that day. The second DataFrame contains the Approach Date, Object Id, Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag. The final dataset containing all NEO observations is titled neos_df.

The neos_df contains 7 columns, 480 rows, and no missing values. The full structure of the DataFrame is as follows:
<pre>
 #   Column                             Non-Null Count  Dtype  
 0   Approach Date                      480 non-null    str 

 1   miss_distance                      480 non-null    str    

 2   relative_velocity                  480 non-null    str    
 
 3   object_id                          480 non-null    str    

 4   absolute_magnitude_h               480 non-null    float64

 5   estimated_diameter                 480 non-null    float64
 
 6   is_potentially_hazardous_asteroid  480 non-null    bool
 <pre>

In [19]:
url = "https://api.nasa.gov/neo/rest/v1/feed"

SESSION = requests.Session()

response = requests.get(url, timeout=10,
                        params={"api_key": "DEMO_KEY",
                                "start_date": "2026-10-21",
                                "end_date": "2026-10-28"})

In [20]:
data = response.json()
print(data.keys())

neos = data['near_earth_objects']

dict_keys(['element_count', 'links', 'near_earth_objects'])


In [21]:
neos.keys()

dict_keys(['2026-10-28', '2026-10-27', '2026-10-22', '2026-10-21', '2026-10-26', '2026-10-25', '2026-10-24', '2026-10-23'])

In [22]:
neos['2026-10-28'][0]

{'absolute_magnitude_h': 21.85,
 'close_approach_data': [{'close_approach_date': '2026-10-28',
   'close_approach_date_full': '2026-Oct-28 16:55',
   'epoch_date_close_approach': 1793206500000,
   'miss_distance': {'astronomical': '0.1530448084',
    'kilometers': '22895177.351198108',
    'lunar': '59.5344304676',
    'miles': '14226403.5189375704'},
   'orbiting_body': 'Earth',
   'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
    'kilometers_per_second': '5.6516180045',
    'miles_per_hour': '12642.1043141844'}}],
 'estimated_diameter': {'feet': {'estimated_diameter_max': 831.8117101206,
   'estimated_diameter_min': 371.997505662},
  'kilometers': {'estimated_diameter_max': 0.2535362011,
   'estimated_diameter_min': 0.1133848361},
  'meters': {'estimated_diameter_max': 253.5362011316,
   'estimated_diameter_min': 113.3848360975},
  'miles': {'estimated_diameter_max': 0.1575400428,
   'estimated_diameter_min': 0.070454049}},
 'id': '2518810',
 'is_potentially_hazard

In [23]:
neos['2026-10-28'][0]['close_approach_data']

[{'close_approach_date': '2026-10-28',
  'close_approach_date_full': '2026-Oct-28 16:55',
  'epoch_date_close_approach': 1793206500000,
  'miss_distance': {'astronomical': '0.1530448084',
   'kilometers': '22895177.351198108',
   'lunar': '59.5344304676',
   'miles': '14226403.5189375704'},
  'orbiting_body': 'Earth',
  'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
   'kilometers_per_second': '5.6516180045',
   'miles_per_hour': '12642.1043141844'}}]

In [24]:
approach_dates = [app_date for app_date in neos]

close_approach_data = {
    "Approach Date": [neos[ad][i]['close_approach_data'][j]['close_approach_date']
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],
    
    "miss_distance": [neos[ad][i]['close_approach_data'][j]['miss_distance']['kilometers'] 
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],

    "relative_velocity": [neos[ad][i]['close_approach_data'][j]['relative_velocity']['kilometers_per_hour'] 
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],
}

close_approach_df = pd.DataFrame(data=close_approach_data)

In [25]:
close_approach_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Approach Date      60 non-null     str  
 1   miss_distance      60 non-null     str  
 2   relative_velocity  60 non-null     str  
dtypes: str(3)
memory usage: 1.5 KB


In [26]:
object_data = {
    "Approach Date": [ad for ad in neos for _ in range(len(neos[ad]))],

    "object_id": [neos[ad][i]['id'] for ad in approach_dates for i in range(len(neos[ad]))],

    "absolute_magnitude_h": [neos[ad][i]['absolute_magnitude_h'] for ad in approach_dates for i in range(len(neos[ad]))],
    
    "estimated_diameter": [neos[ad][i]['estimated_diameter']['kilometers']['estimated_diameter_min'] 
                           for ad in approach_dates
                              for i in range(len(neos[ad]))],

    "is_potentially_hazardous_asteroid": [neos[ad][i]['is_potentially_hazardous_asteroid']
                                          for ad in approach_dates
                                            for i in range(len(neos[ad]))],
}

object_data_df = pd.DataFrame(data=object_data)

In [27]:
object_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Approach Date                      60 non-null     str    
 1   object_id                          60 non-null     str    
 2   absolute_magnitude_h               60 non-null     float64
 3   estimated_diameter                 60 non-null     float64
 4   is_potentially_hazardous_asteroid  60 non-null     bool   
dtypes: bool(1), float64(2), str(2)
memory usage: 2.1 KB


In [28]:
neos_df = close_approach_df.merge(object_data_df, on="Approach Date", how="left", validate="many_to_many")
neos_df['Approach Date'] = pd.to_datetime(neos_df['Approach Date'])
neos_df = neos_df.sort_values(by='object_id')
display(neos_df.head())

,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
210,2026-10-26,66425278.809385698,67073.2296998069,2488509,20.61,0.200704,False
282,2026-10-26,46355017.547430276,32270.11064715,2488509,20.61,0.200704,False
273,2026-10-26,45093384.613895265,73474.2645370801,2488509,20.61,0.200704,False
264,2026-10-26,21164664.944929038,12565.2958386346,2488509,20.61,0.200704,False
246,2026-10-26,64867720.105932295,71372.1280550965,2488509,20.61,0.200704,False


In [36]:
type(neos_df.groupby(by='object_id')['absolute_magnitude_h'])

pandas.api.typing.SeriesGroupBy

In [37]:
neos_df.groupby(by='object_id')['absolute_magnitude_h'].first()

object_id
2488509    20.61
2516803    19.52
2518810    21.85
2529819    20.86
3092205    19.49
3114023    26.10
3114028    20.80
3137852    18.54
3159894    19.58
3180184    23.60
3328821    21.50
3387458    21.88
3388488    22.00
3389488    23.00
3390647    22.13
3479014    23.00
3485264    27.20
3576729    26.00
3615557    22.90
3617076    28.70
3648711    25.00
3671134    22.37
3683469    22.05
3715996    27.00
3732669    26.02
3753695    19.88
3757572    20.91
3765640    24.04
3774725    23.80
3781448    21.85
3785687    24.95
3786454    24.63
3788075    26.10
3788085    25.60
3788088    25.25
3789125    25.10
3789972    23.21
3799748    23.28
3799826    25.72
3802406    27.80
3802428    22.59
3802572    22.16
3831888    19.91
3835500    24.08
3836049    27.50
3836242    26.90
3837949    27.64
3838032    26.10
3838249    25.53
3840124    26.30
3841527    24.20
3842752    27.00
3873149    25.16
3879285    24.30
3879547    25.60
3879718    22.66
3879723    25.72
3884012    27.20
3986

## Part 2

In [ ]:
def group_neos_by_hazardous_flag(df: pd.DataFrame, hazardous_flag: bool) -> pd.DataFrame:
    """Group neos as hazardous or not hazardous asteriods.

    Args:
        df (pd.DataFrame): Pandas DataFrame to be filtered.
        hazardous_flag (bool): Flag to indicate hazardous or not hazardous. True for hazardous, False not hazardous.

    Returns:
        pd.DataFrame: Pandas DataFrame containing only potentially hazardous asteriods or not hazardous asteriods based on flag.
    """

    return df.loc[df['is_potentially_hazardous_asteroid'] == hazardous_flag]

def filter_by_date(df: pd.DataFrame, start_date: str, end_date: str, data_column: str ="Approach Date") -> pd.DataFrame:
    """Filter NEO DataFrame by start and end dates.

    Args:
        df (pd.DataFrame): Pandas DataFrame to be filtered.
        start_date (str): Beginning of date filter (YYYY-MM-DD)
        end_date (str): End of date filter (YYYY-MM-DD)

    Returns:
        pd.DataFrame: Pandas DataFrame containing observations between start and end date.
    """
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    return df[df[data_column].between(start_date, end_date)]

def get_summary_dictionary(grouped_column: pd.api.typing.DataFrameGroupBy):
    
    return {
        'Measurement': grouped_column.first().Name(),
        'mean': grouped_column.first().mean(),
        'median': grouped_column.first().median(),
        'mode': grouped_column.first().mode(),
        'count': len(grouped_column.first()),
        'sd': grouped_column.first().std()
    }

def summarize_potentialy_hazardous_neos(df: pd.DataFrame):
    hazardous_neos = group_neos_by_hazardous_flag(df=df, hazardous_flag=True)
    nonhazardous_neos = group_neos_by_hazardous_flag(df=df, hazardous_flag=False)


